# Final Model Comparison and Selection

## Scientific objective
Apply a predefined reliability-oriented decision framework using PR-AUC, operational recall/MCC, calibration, AD coverage, OOD/cliff/external evidence, compute, interpretability, and reproducibility.

## Inputs
- all model metrics
- calibration and AD summaries
- cliff/external results

## Expected outputs
- `tables/final_model_comparison.csv`
- `reports/final_model_selection.json`

## Dependencies
pandas

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
ROC-AUC alone cannot select the final model. A simpler model is preferred when reliability is comparable.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
A valid final selection requires full-profile repeated runs. Smoke-mode ranking is marked provisional.

## Next notebook
[23_candidate_screening_demo.ipynb](./23_candidate_screening_demo.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
from toxicity_screening.utils import atomic_write_json
metrics=pd.read_csv(ROOT/"results/ablations/ablation_matrix.csv")
cal=pd.read_csv(ROOT/"results/calibration/calibrated_model_summary.csv")
ad_path=ROOT/"results/applicability_domain/ad_performance_summary.csv"
ad=pd.read_csv(ad_path) if ad_path.exists() else pd.DataFrame()
# Score is deliberately transparent and only ranks candidates with available fields.
comparison=metrics.copy()
for col in ["pr_auc","mcc","balanced_accuracy","brier","ece"]:
    if col not in comparison: comparison[col]=np.nan
comparison["reliability_score"]=(comparison["pr_auc"].fillna(0)+comparison["mcc"].fillna(0)+comparison["balanced_accuracy"].fillna(0)-comparison["brier"].fillna(1)-comparison["ece"].fillna(1))
comparison.to_csv(ROOT/"tables/final_model_comparison.csv",index=False)
selection=[]
for endpoint,g in comparison.groupby("endpoint"):
    best=g.sort_values("reliability_score",ascending=False).iloc[0]
    selection.append({"endpoint":endpoint,"provisional_family":best.get("family"),"provisional_model":best.get("model"),"score":best.reliability_score,"profile":PROFILE,"selection_status":"provisional" if PROFILE=="smoke" else "requires external/cliff review"})
atomic_write_json({"framework":{"positive":["pr_auc","mcc","balanced_accuracy","calibration","AD coverage","external/cliff robustness"],"penalties":["brier","ece","compute","unsupported OOD confidence"]},"selections":selection},ROOT/"reports/final_model_selection.json")
display(pd.DataFrame(selection))

,endpoint,provisional_family,provisional_model,score,profile,selection_status
0,SR-ARE,qsar,svm,1.324830,full,requires external/cliff review
1,SR-ATAD5,qsar,svm,0.846759,full,requires external/cliff review
2,SR-MMP,qsar,svm,1.696185,full,requires external/cliff review
3,SR-p53,qsar,svm,1.082973,full,requires external/cliff review
4,ames_mutagenicity,qsar,svm,2.083349,full,requires external/cliff review
5,herg_blockade,qsar,random_forest,1.884548,full,requires external/cliff review


### Completion gate
Confirm that the declared artifacts exist before continuing to `23_candidate_screening_demo.ipynb`.